In [2]:
import cv2
import numpy as np
from pdf2image import convert_from_path
from scipy.ndimage import rotate
from PIL import Image
import pytesseract

# Function to correct skew
def correct_skew(image, delta=1, limit=12):
    def determine_score(arr, angle):
        data = rotate(arr, angle, reshape=False, order=0)
        histogram = np.sum(data, axis=1, dtype=float)
        score = np.sum((histogram[1:] - histogram[:-1]) ** 2, dtype=float)
        return histogram, score

    gray = cv2.cvtColor(np.array(image), cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 41, 15)
    scores = []
    angles = np.arange(-limit, limit + delta, delta)
    
    for angle in angles:
        histogram, score = determine_score(thresh, angle)
        scores.append(score)
    
    best_angle = angles[scores.index(max(scores))]
    return best_angle

# Function to deskew image
def deskew_image(image, angle):
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return rotated

# Function to detect text orientation using Tesseract
def detect_orientation(image):
    image = np.array(image)
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    osd = pytesseract.image_to_osd(rgb_image)
    rotate_angle = 0
    if "Rotate: 180" in osd:
        rotate_angle = 180
    return rotate_angle

# Path to the PDF file
pdf_path = '/home/emanmunir/detection/test2/pdf to image/scanned.pdf'
# Convert PDF to images
images = convert_from_path(pdf_path)

# Process images and save to a new PDF
pdf_images = []
for i, image in enumerate(images):
    rotate_angle = detect_orientation(image)
    if rotate_angle == 180:
        image = image.rotate(180, expand=True)
    angle = correct_skew(image)
    deskewed_image = deskew_image(np.array(image), angle)
    final_image = Image.fromarray(deskewed_image)
    pdf_images.append(final_image.convert('RGB'))

# Save all images into a single PDF
pdf_images[0].save('/home/emanmunir/detection/test2/pdf to image/corrected.pdf', save_all=True, append_images=pdf_images[1:])

print("PDF pages have been converted, deskewed, and saved as a single PDF.")


PDF pages have been converted, deskewed, and saved as a single PDF.
